# DLGenAI Project — Milestone 4
**Roll No:** 23f3004491

Multiple-choice formulation of the Smart MCQ Solver: label encoding, prompt-option
formatting, MC tokenization, AutoModelForMultipleChoice logits, LoRA, and a tiny
Trainer fine-tune. Each cell prints the answer for the corresponding form question.

In [1]:
!pip install -q -U "transformers==4.46.3" "peft==0.13.2" "accelerate==1.1.1"
print("ready - restart kernel if imports fail")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 80.9 MB/s eta 0:00:00
ready - restart kernel if imports fail


In [2]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import torch

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
OPTIONS = ['A', 'B', 'C', 'D', 'E']
print(train.shape)

(2000, 8)


## Q1. Label Encoding
A=0, B=1, C=2, D=3, E=4 — encoded label at row index 150.

In [3]:
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
train['label'] = train['answer'].map(label_map)

print("Q1 answer:", train.loc[150, 'label'])

Q1 answer: 2


## Q2. Prompt-Option Formatting
`str(prompt) + " [SEP] " + str(option_B)` for row 0 — exact character length.

In [4]:
row0 = train.iloc[0]
formatted = str(row0['prompt']) + " [SEP] " + str(row0['B'])

print("Q2 answer:", len(formatted))

Q2 answer: 407


## Q3. Single-Row MCQ Tokenization
Shape [1, 5, 128] — value of the second dimension.

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

inputs_row0 = [str(row0['prompt']) + " [SEP] " + str(row0[o]) for o in OPTIONS]
tok = tokenizer(inputs_row0, padding="max_length", truncation=True,
                max_length=128, return_tensors="pt")
input_ids_row0 = tok['input_ids'].unsqueeze(0)

print("shape:", tuple(input_ids_row0.shape))
print("Q3 answer:", input_ids_row0.shape[1])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

shape: (1, 5, 128)
Q3 answer: 5


## Q4. Batch MCQ Tokenization
16 rows x 5 choices x 128 tokens — total token positions.

In [6]:
batch = train.head(16)
flat = [str(r['prompt']) + " [SEP] " + str(r[o]) for _, r in batch.iterrows() for o in OPTIONS]
tok16 = tokenizer(flat, padding="max_length", truncation=True,
                  max_length=128, return_tensors="pt")
ids16 = tok16['input_ids'].view(16, 5, 128)

total = ids16.numel()
print("shape:", tuple(ids16.shape))
print("Q4 answer:", total)

shape: (16, 5, 128)
Q4 answer: 10240


## Q5. Multiple-Choice Logits
Pass row 0 through AutoModelForMultipleChoice — logits per question.

In [7]:
from transformers import AutoModelForMultipleChoice

mc_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")
mc_model.eval()

with torch.no_grad():
    out = mc_model(input_ids=input_ids_row0,
                   attention_mask=tok['attention_mask'].unsqueeze(0))

print("logits shape:", tuple(out.logits.shape))
print("Q5 answer:", out.logits.shape[1])

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


logits shape: (1, 5)
Q5 answer: 5


## Q6. Supervised Loss Tensor
Pass the same input with the correct label — dimensions of the loss tensor.

In [8]:
label0 = torch.tensor([train.loc[0, 'label']])

with torch.no_grad():
    out_l = mc_model(input_ids=input_ids_row0,
                     attention_mask=tok['attention_mask'].unsqueeze(0),
                     labels=label0)

print("loss:", out_l.loss.item())
print("Q6 answer:", out_l.loss.dim())

loss: 1.6125731468200684
Q6 answer: 0


## Q7. LoRA Trainable Parameters
r=8, alpha=16, target=[query,value], dropout=0.1, bias=none, SEQ_CLS.

In [9]:
from peft import LoraConfig, get_peft_model, TaskType

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

mc_lora = get_peft_model(AutoModelForMultipleChoice.from_pretrained("bert-base-uncased"), lora_cfg)

trainable = sum(p.numel() for p in mc_lora.parameters() if p.requires_grad)
print("Q7 answer:", trainable)
mc_lora.print_trainable_parameters()

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Q7 answer: 295681
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


## Q8. Hugging Face Dataset Preparation
First 100 rows -> input_ids [5,128] per item — choices stored per item.

In [10]:
from datasets import Dataset

def encode_row(r):
    texts = [str(r['prompt']) + " [SEP] " + str(r[o]) for o in OPTIONS]
    t = tokenizer(texts, padding="max_length", truncation=True, max_length=128)
    return {'input_ids': t['input_ids'],
            'attention_mask': t['attention_mask'],
            'labels': int(r['label'])}

ds100 = Dataset.from_list([encode_row(r) for _, r in train.head(100).iterrows()])
ds100.set_format('torch')

item0 = ds100[0]
print("input_ids shape:", tuple(item0['input_ids'].shape))
print("Q8 answer:", item0['input_ids'].shape[0])

input_ids shape: (5, 128)
Q8 answer: 5


## Q9. Tiny LoRA Fine-Tuning
32 rows, max_length=64, batch=4, grad_accum=1, max_steps=4 — final global_step.

In [11]:
from transformers import TrainingArguments, Trainer

def encode_row64(r):
    texts = [str(r['prompt']) + " [SEP] " + str(r[o]) for o in OPTIONS]
    t = tokenizer(texts, padding="max_length", truncation=True, max_length=64)
    return {'input_ids': t['input_ids'],
            'attention_mask': t['attention_mask'],
            'labels': int(r['label'])}

ds32 = Dataset.from_list([encode_row64(r) for _, r in train.head(32).iterrows()])
ds32.set_format('torch')

args = TrainingArguments(
    output_dir="./m4_tiny",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to="none",
)

tiny_trainer = Trainer(model=mc_lora, args=args, train_dataset=ds32)
result = tiny_trainer.train()

print("Q9 answer:", tiny_trainer.state.global_step)

max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss
1,1.622600
2,1.572800
3,1.586200
4,1.705500


Q9 answer: 4


## Q10. Probability of Option E After Fine-Tuning
Row 0 through the fine-tuned model, softmax over logits — P(E) to 4 decimals.

In [12]:
ds1 = Dataset.from_list([encode_row64(train.iloc[0])])
ds1.set_format('torch')

mc_lora.eval()
with torch.no_grad():
    out10 = mc_lora(input_ids=ds1[0]['input_ids'].unsqueeze(0).to(mc_lora.device),
                    attention_mask=ds1[0]['attention_mask'].unsqueeze(0).to(mc_lora.device))
probs = torch.softmax(out10.logits[0].float(), dim=-1)

for o, p in zip(OPTIONS, probs.tolist()):
    print(f"P({o}) = {p:.4f}")
print("Q10 answer:", round(probs[4].item(), 4))

P(A) = 0.2026
P(B) = 0.2091
P(C) = 0.2008
P(D) = 0.1938
P(E) = 0.1937
Q10 answer: 0.1937
